# 02_hypothetical_personas.ipynb

-----

In [1]:
%load_ext autoreload
%autoreload 2

In [12]:
import yfinance as yf
import pandas as pd

from src.data.utils import *

In [3]:
# Read the dictionary with the cleaned data
import pickle

with open("cleaned_data.pkl", "rb") as f:
    cleaned_data = pickle.load(f)

In [4]:
data_10y_1d = timeframe_selector(cleaned_data,
                   stocks=["AAPL", "MSFT", "IBM", "XOM", "WTM", "KO", "JNJ", "VZ", "KHC", "USB"],
                   timeframe='10y', freq='1d')

data_close_10y_1d = close_prices_df_generator(data_10y_1d)
returns_close_10y_1d = compute_returns(data_close_10y_1d)

## Import risk free rate

In [78]:
risk_free_rate = pd.read_csv('../data/processed/risk_free.csv',
                             index_col='observation_date',
                             parse_dates=True)

In [91]:
risk_free_rate + 1

,DTB3
observation_date,
2015-01-02,1.0002
2015-01-05,1.0003
2015-01-06,1.0003
2015-01-07,1.0003
2015-01-08,1.0003
...,...
2025-11-07,1.0377
2025-11-10,1.0379
2025-11-11,1.0379


In [ ]:
# Compute the daily risk free rate
daily_rf_rate = (1 + risk_free_rate)**(1/252) - 1
daily_rf_rate

,DTB3
observation_date,
2015-01-02,7.935718e-07
2015-01-05,1.190298e-06
2015-01-06,1.190298e-06
2015-01-07,1.190298e-06
2015-01-08,1.190298e-06
...,...
2025-11-07,1.468629e-04
2025-11-10,1.476277e-04
2025-11-11,1.476277e-04


## 1. Buy and Hold

### 1.1. Equally weighted portfolio

In [74]:
n_assets = data_close_10y_1d.shape[1]
weights = np.full(n_assets, 1/n_assets)

portfolio_equally_ret = returns_close_10y_1d @ weights
portfolio_equally_ret = pd.DataFrame(portfolio_equally_ret, columns=['portfolio returns'])

In [ ]:
pd.merge(portfolio_equally_ret,
         daily_rf_rate,
         right_on='observation_date',
         how='inner',
         left_index=True)

,portfolio returns,DTB3
observation_date,,
2015-10-06,-0.000116,0.000000e+00
2015-10-07,0.008995,0.000000e+00
2015-10-08,0.007106,-3.968452e-07
2015-10-09,0.000850,3.968056e-07
2015-10-12,-0.000269,3.968056e-07
...,...,...
2025-09-29,-0.007616,1.503036e-04
2025-09-30,0.005087,1.503036e-04
2025-10-01,0.001016,1.499214e-04


In [36]:
# Start with $1 and see how it grows
wealth_index = (1 + portfolio_equally_ret).cumprod()

In [97]:
wealth_index[-1]-1

C:\Users\JUANJO\AppData\Local\Temp\ipykernel_1864\1098091611.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  wealth_index[-1]-1


np.float64(2.2543533679307233)

In [ ]:
def performance_metrics(weights: np.array, returns: pd.DataFrame, daily_rf_rate: pd.DataFrame) -> dict:
    """
    Calculates key performance metrics for a given portfolio.
    
    Args:
    weights (np.array): Series of asset weights (N x 1)
    returns (pd.DataFrame): DataFrame of assets returns (T x N)
    daily_rf_rate (pd.DataFrame): DataFrame of daily risk free rate DTB3 (T x 1)
    
    Returns:
    dict: A dictionary containing the wealth series and key metrics.
    """

    # Calculate portfolio returns
    portfolio_ret = returns @ weights
    portfolio_ret = pd.DataFrame(portfolio_ret, columns=['return'])

    # Calculate Wealth index
    wealth_index = (1 + portfolio_ret).cumprod()

    # Merge datasets aligning them by date
    portfolio_rf = pd.merge(portfolio_ret,
         daily_rf_rate,
         right_on='observation_date',
         how='inner',
         left_index=True)
    
    # Excess returns
    portfolio_rf['excess_ret'] = portfolio_rf['return'] - portfolio_rf['DTB3']
    excess_returns = portfolio_rf['excess_ret']

    # Calculate annualized sharpe ratio from excess returns
    mean_excess_return = excess_returns.mean()
    std_excess_return = excess_returns.std()

    # Avoid division by zero if std is 0
    if std_excess_return == 0:
        annualized_sharpe = 0.0
    else:
        daily_sharpe = mean_excess_return / std_excess_return
        annualized_sharpe = daily_sharpe * np.sqrt(252)
     
    # Calculate Max Drawdown
    running_peak = wealth_index.cummax()
    drawdown = (wealth_index - running_peak) / running_peak
    max_drawdown = drawdown.min()

    return {
        "wealth": wealth_index,
        "total_return": wealth_index.iloc[-1] - 1,
        "annualized_sharpe": annualized_sharpe,
        "max_drawdown": max_drawdown,
        "portfolio_daily_returns": portfolio_returns
    }
    

### 1.2. Market Cap weighted portfolio

In [22]:
ticker_symbols = data_close_10y_1d.columns

# Get market capitalization data
mktcap_data = {}
for t in ticker_symbols:
    ticker = yf.Ticker(t)
    market_cap = ticker.fast_info['market_cap']
    mktcap_data[t] = market_cap

mktcap_data

{'AAPL': 4033205605452.029,
 'MSFT': 3791850377591.54,
 'IBM': 285739207404.2084,
 'XOM': 503045933860.7788,
 'WTM': 4913886637.217041,
 'KO': 306102501163.1621,
 'JNJ': 472053171688.65564,
 'VZ': 173337257631.85037,
 'KHC': 29544043713.377487,
 'USB': 73540273339.0466}

In [32]:
# Total market capitalization
total_mktcap = sum(mktcap_data.values())

weights_mktcap = {key: value / total_mktcap for key, value in mktcap_data.items()}

In [33]:
weights_mktcap
    

{'AAPL': 0.4169406671538233,
 'MSFT': 0.391990089482012,
 'IBM': 0.02953886000903812,
 'XOM': 0.05200337592246903,
 'WTM': 0.0005079828239031486,
 'KO': 0.03164395608294777,
 'JNJ': 0.04879943736810502,
 'VZ': 0.017919084447850234,
 'KHC': 0.0030541743965707777,
 'USB': 0.007602372313280597}